# ARCHS4 CLAMP models with all pathways prior at 5% study coverage

**Environment:** `clamp-analyses`

Self-contained pipeline: study-level sampling targeting 5% of total ARCHS4 samples (selecting whole GEO studies) → SVD → CLAMPbase → CLAMPfull with all pathways prior (Hallmark, Reactome, GO CC, C8).

Steps (repeated for each seed):
1. Sample 5% of unique GEO studies (series_id) from ARCHS4
2. Take all samples belonging to those studies
3. Compute SVD
4. Run CLAMPbase
5. Run CLAMPfull with all pathways prior
6. Save results to `study_coverage_rs5_seed_*`

## Load libraries

In [ ]:
library(bigstatsr)
library(data.table)
library(dplyr)
library(rsvd)
library(Matrix)
library(here)
library(rhdf5)
library(CLAMP)
library(PCAtools)

source(here("config.R"))

## Configuration

In [ ]:
base_output_dir <- config$ARCHS4$DATASET_FOLDER
archs4_file     <- config$ARCHS4$DATASET_FILE

coverage     <- 0.05
coverage_pct <- as.integer(coverage * 100)

pathways_path <- here::here('data/pathways')

N_CORES    <- config$ARCHS4$CLAMP_PARAMS$RANDOM_SVD_N_CORES
MULTIPLIER <- 100
MAX_ITER   <- 5000

base_seed <- config$ARCHS4$CLAMP_PARAMS$RANDOM_SVD_SEED
seeds     <- base_seed + 0:2
n_runs    <- length(seeds)
message("Coverage: ", coverage_pct, "% of samples (study-based) — will run ", n_runs, " seeds")

## Load preprocessed data, pathways, and study mapping

In [ ]:
# metadata
meta_file <- file.path(base_output_dir, "metadata_filtered.rds")
if (!file.exists(meta_file))
  meta_file <- file.path(base_output_dir, "01_archs4_preprocess", "metadata_filtered.rds")
meta            <- readRDS(meta_file)
n_genes_thin    <- meta$n_genes_thin
n_samples_total <- meta$n_samples
archs4_genes    <- meta$gene_symbols_thin

# all_samples
samples_file <- file.path(base_output_dir, "all_samples.rds")
if (!file.exists(samples_file))
  samples_file <- file.path(base_output_dir, "01_archs4_preprocess", "all_samples.rds")
all_samples        <- readRDS(samples_file)
sample_names_total <- all_samples[seq_len(n_samples_total)]

# Build GSM → GSE mapping from ARCHS4 HDF5
message("Reading series information from ARCHS4 HDF5 file...")
geo_acc   <- rhdf5::h5read(archs4_file, "/meta/samples/geo_accession")
series_id <- rhdf5::h5read(archs4_file, "/meta/samples/series_id")
gsm_to_gse <- data.frame(
  geo_accession = geo_acc,
  series_id     = series_id,
  stringsAsFactors = FALSE
)

sample_series_df <- data.frame(
  sample_idx  = seq_len(n_samples_total),
  sample_name = sample_names_total,
  stringsAsFactors = FALSE
) %>%
  dplyr::left_join(gsm_to_gse, by = c("sample_name" = "geo_accession"))

study_sizes <- sample_series_df %>%
  dplyr::filter(!is.na(series_id)) %>%
  dplyr::group_by(series_id) %>%
  dplyr::summarise(n = dplyr::n(), .groups = "drop")
n_studies_total  <- nrow(study_sizes)
n_samples_target <- round(n_samples_total * coverage)
message("Total studies: ", n_studies_total,
        " | sample target: ", n_samples_target,
        " (", coverage_pct, "% of ", n_samples_total, ")")

hall_gmt     <- CLAMP:::read_gmt(file.path(pathways_path, "h.all.v2026.1.Hs.symbols.gmt"))
reactome_gmt <- CLAMP:::read_gmt(file.path(pathways_path, "c2.cp.reactome.v2026.1.Hs.symbols.gmt"))
gocc_gmt     <- CLAMP:::read_gmt(file.path(pathways_path, "c5.go.cc.v2026.1.Hs.symbols.gmt"))
c8_gmt       <- CLAMP:::read_gmt(file.path(pathways_path, "c8.all.v2026.1.Hs.symbols.gmt"))

names(hall_gmt)     <- paste0("HALL_",     names(hall_gmt))
names(reactome_gmt) <- paste0("REACTOME_", names(reactome_gmt))
names(gocc_gmt)     <- paste0("GOCC_",     names(gocc_gmt))
names(c8_gmt)       <- paste0("C8_",       names(c8_gmt))

all_pathways_list <- list(
  HALL     = hall_gmt,
  REACTOME = reactome_gmt,
  GOCC     = gocc_gmt,
  C8       = c8_gmt
)

all_pathways_pathMat <- gmtListToSparseMat(all_pathways_list)
all_pathways_matched <- getMatchedPathwayMat(all_pathways_pathMat, archs4_genes)
message("Loaded and matched all pathways matrix")

# FBM
fbm_backing <- file.path(base_output_dir, "fbm_filtered")
if (!file.exists(paste0(fbm_backing, ".bk")))
  fbm_backing <- file.path(base_output_dir, "01_archs4_preprocess", "fbm_filtered")
archs4_fbm_filt <- FBM(
  nrow        = n_genes_thin,
  ncol        = n_samples_total,
  backingfile = fbm_backing,
  create_bk   = FALSE
)
message("Loaded FBM with ", n_genes_thin, " genes and ", n_samples_total, " samples")

## Run 5% study coverage models (SVD → CLAMPbase → CLAMPfull)

In [ ]:
for (run_idx in seq_len(n_runs)) {
  current_seed <- seeds[run_idx]
  message("\n", strrep("=", 60))
  message("RUN ", run_idx, "/", n_runs, " - Seed: ", current_seed)
  message(strrep("=", 60))

  dst_dir <- file.path(base_output_dir, "07_bp_coverage_study", "01_bp_coverage_study_05",
    paste0("study_coverage_rs", coverage_pct, "_seed_", run_idx))
  dir.create(dst_dir, showWarnings = FALSE, recursive = TRUE)

  set.seed(current_seed)
  shuffled <- study_sizes[sample(nrow(study_sizes)), ]
  cum_n    <- cumsum(shuffled$n)
  n_take   <- which(cum_n >= n_samples_target)[1]
  if (is.na(n_take)) n_take <- nrow(shuffled)
  selected_series <- shuffled$series_id[seq_len(n_take)]
  sample_idx      <- sort(sample_series_df$sample_idx[
                             sample_series_df$series_id %in% selected_series])
  n_samples    <- length(sample_idx)
  sample_names <- sample_names_total[sample_idx]
  message("Selected ", n_take, " studies → ", n_samples,
          " samples (target: ", n_samples_target, ")")

  saveRDS(list(
    run              = run_idx,
    seed             = current_seed,
    coverage         = coverage,
    n_studies_total  = n_studies_total,
    n_studies        = n_take,
    study_ids        = selected_series,
    n_samples        = n_samples,
    n_samples_target = n_samples_target,
    sample_idx       = sample_idx,
    sample_names     = sample_names,
    sampling_method  = "study_sampling"
  ), file = file.path(dst_dir, "subsample_info.rds"))

  message("Creating subsampled FBM...")
  unlink(paste0(file.path(dst_dir, "fbm_subsampled"), c(".bk", ".rds")), force = TRUE)
  Y_sub <- big_copy(
    archs4_fbm_filt,
    ind.col     = sample_idx,
    backingfile = file.path(dst_dir, "fbm_subsampled")
  )

  message("Computing SVD...")
  SVD_K <- round(min(n_samples - 1, n_genes_thin - 1) / 4)

  if (N_CORES > 1) {
    options(bigstatsr.check.parallel.blas = FALSE)
    blas_nproc <- getOption("default.nproc.blas")
    options(default.nproc.blas = NULL)
  }

  svd_result <- big_randomSVD(Y_sub, k = SVD_K, ncores = N_CORES)

  if (N_CORES > 1) {
    options(bigstatsr.check.parallel.blas = TRUE)
    options(default.nproc.blas = blas_nproc)
  }

  valid_idx    <- which(!is.nan(svd_result$d))
  svd_result$d <- svd_result$d[valid_idx]
  svd_result$u <- svd_result$u[, valid_idx, drop = FALSE]
  svd_result$v <- svd_result$v[, valid_idx, drop = FALSE]
  saveRDS(svd_result, file = file.path(dst_dir, "svd.rds"))

  eigenvalues <- sort(svd_result$d^2 / (n_samples - 1), decreasing = TRUE)
  noise_gd    <- median(eigenvalues)
  CLAMP_K     <- PCAtools::chooseGavishDonoho(
    .dim          = c(n_genes_thin, n_samples),
    var.explained = eigenvalues,
    noise         = noise_gd
  ) * 2
  message("CLAMP_K (Gavish-Donoho) = ", CLAMP_K)
  saveRDS(CLAMP_K, file = file.path(dst_dir, "CLAMP_K.rds"))

  message("Running CLAMPbase...")
  baseRes <- CLAMPbase(Y = Y_sub, svdres = svd_result, trace = TRUE, clamp_k = CLAMP_K)
  baseRes$Z <- data.frame(baseRes$Z)
  rownames(baseRes$Z) <- archs4_genes
  baseRes$B <- data.frame(baseRes$B)
  colnames(baseRes$B) <- sample_names
  saveRDS(baseRes, file = file.path(dst_dir, "CLAMPbase.rds"))

  model_dir <- file.path(dst_dir, "CLAMPbase")
  dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)
  write.csv(baseRes$B, file.path(model_dir, "B.csv"))
  write.csv(baseRes$Z, file.path(model_dir, "Z.csv"))

  message("Running CLAMPfull with hall prior...")
  fullRes <- CLAMPfull(
    Y                 = Y_sub,
    svdres            = svd_result,
    priorMat          = all_pathways_matched,
    clamp.base.result = baseRes,
    use_cpp           = TRUE,
    trace             = TRUE,
    multiplier        = MULTIPLIER,
    max.iter          = MAX_ITER,
    clamp_k           = CLAMP_K
  )
  fullRes$Z <- data.frame(fullRes$Z)
  rownames(fullRes$Z) <- archs4_genes
  fullRes$B <- data.frame(fullRes$B)
  colnames(fullRes$B) <- sample_names
  fullRes$summary <- fullRes$summary %>%
    dplyr::rename(LV = LV_index) %>%
    dplyr::mutate(LV = paste0('LV', LV))

  saveRDS(fullRes, file = file.path(dst_dir, "CLAMPfull_hall.rds"))
  model_dir <- file.path(dst_dir, "CLAMPfull_hall")
  dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)
  write.csv(fullRes$B,       file.path(model_dir, "B.csv"))
  write.csv(fullRes$Z,       file.path(model_dir, "Z.csv"))
  write.csv(fullRes$summary, file.path(model_dir, "summary.csv"))

  rm(Y_sub, svd_result, baseRes, fullRes)
  gc()
}

message("\n", strrep("=", 60))
message("All ", n_runs, " runs completed!")
message(strrep("=", 60))